#### **5.1.5. Entrenamiento K-fold y mejor modelo**

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import Subset, ConcatDataset
import numpy as np

K_FOLDS          = 5
MAX_EPOCHS_KFOLD = 50
PATIENCE_KFOLD   = 5

full_dataset = ConcatDataset([train_dataset_mrcnn, val_dataset_mrcnn])
kf           = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
all_indices  = list(range(len(full_dataset)))
fold_results = []

print(f"\n{'='*60}")
print(f"VALIDACIÓN CRUZADA K-FOLD (k={K_FOLDS}) — Mask R-CNN")
print(f"  LR         : {best_hp['lr']}")
print(f"  Batch size : {best_hp['batch_size']}")
print(f"  Optimizer  : {best_hp['optimizer']}")
print(f"  Scheduler  : {best_hp['scheduler']}")
print(f"{'='*60}")

for fold, (train_idx, val_idx) in enumerate(kf.split(all_indices)):
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold+1}/{K_FOLDS} | "
          f"Train: {len(train_idx)} | Val: {len(val_idx)}")
    print(f"{'─'*60}")

    train_loader_fold = DataLoader(
        Subset(full_dataset, train_idx), batch_size=best_hp['batch_size'],
        shuffle=True, collate_fn=collate_fn, num_workers=0
    )
    val_loader_fold = DataLoader(
        Subset(full_dataset, val_idx), batch_size=1,
        shuffle=False, collate_fn=collate_fn, num_workers=0
    )

    model_fold = build_maskrcnn().to(DEVICE)

    if best_hp['optimizer'] == "SGD":
        opt_fold = optim.SGD(model_fold.parameters(),
                             lr=best_hp['lr'], momentum=0.9, weight_decay=1e-4)
    else:
        opt_fold = optim.Adam(model_fold.parameters(),
                              lr=best_hp['lr'], weight_decay=1e-4)

    if best_hp['scheduler'] == "StepLR":
        sch_fold = optim.lr_scheduler.StepLR(opt_fold, step_size=3, gamma=0.5)
    else:
        sch_fold = optim.lr_scheduler.ReduceLROnPlateau(opt_fold, mode='min',
                                                         patience=2, factor=0.5)

    best_val_fold    = float("inf")
    epochs_no_improv = 0
    best_epoch_fold  = 0

    for epoch in range(MAX_EPOCHS_KFOLD):
        model_fold.train()
        train_loss = 0.0
        for imgs, targets in train_loader_fold:
            imgs    = [img.to(DEVICE) for img in imgs]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            opt_fold.zero_grad()
            loss_dict = model_fold(imgs, targets)
            losses    = sum(loss for loss in loss_dict.values())
            losses.backward()
            opt_fold.step()
            train_loss += losses.item()
        train_loss /= len(train_loader_fold)

        model_fold.train()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, targets in val_loader_fold:
                imgs    = [img.to(DEVICE) for img in imgs]
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
                loss_dict = model_fold(imgs, targets)
                losses    = sum(loss for loss in loss_dict.values())
                val_loss += losses.item()
        val_loss /= len(val_loader_fold)

        if best_hp['scheduler'] == "StepLR":
            sch_fold.step()
        else:
            sch_fold.step(val_loss)

        # Early stopping 
        if val_loss < best_val_fold:
            best_val_fold    = val_loss
            best_epoch_fold  = epoch + 1
            epochs_no_improv = 0
            torch.save(model_fold.state_dict(),
                       os.path.join(MODELS_PATH, f"maskrcnn_fold{fold+1}.pth"))
        else:
            epochs_no_improv += 1

        print(f"  Época {epoch+1:02d}/{MAX_EPOCHS_KFOLD} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Sin mejora: {epochs_no_improv}/{PATIENCE_KFOLD}"
              + (" ✓" if epochs_no_improv == 0 else ""))

        if epochs_no_improv >= PATIENCE_KFOLD:
            print(f"\n Early stopping en época {epoch+1}")
            break

    fold_results.append({
        "fold":          fold + 1,
        "best_val_loss": best_val_fold,
        "best_epoch":    best_epoch_fold,
        "total_epochs":  epoch + 1,
    })
    print(f"\n Fold {fold+1} | Val Loss: {best_val_fold:.4f} | "
          f"Época óptima: {best_epoch_fold}/{epoch+1}")

    del model_fold
    torch.cuda.empty_cache()

# Resumen
val_losses = [r['best_val_loss'] for r in fold_results]

print(f"\n{'='*60}")
print("RESUMEN K-FOLD — Mask R-CNN")
print(f"{'='*60}")
print(f"{'Fold':>5} {'Best Ep':>8} {'Total Ep':>9} {'Val Loss':>10}")
print(f"{'─'*60}")
for r in fold_results:
    print(f"  {r['fold']:>3}   {r['best_epoch']:>8}   {r['total_epochs']:>7}   "
          f"{r['best_val_loss']:>9.4f}")
print(f"\n  Media : {np.mean(val_losses):.4f}")
print(f"  Std   : {np.std(val_losses):.4f}")
print(f"  Min   : {np.min(val_losses):.4f}")
print(f"  Max   : {np.max(val_losses):.4f}")

# Cargar mejor fold como modelo final 
best_fold    = min(fold_results, key=lambda x: x["best_val_loss"])
model_maskrcnn = build_maskrcnn().to(DEVICE)
model_maskrcnn.load_state_dict(
    torch.load(os.path.join(MODELS_PATH, f"maskrcnn_fold{best_fold['fold']}.pth"))
)
print(f"\n Mejor fold: Fold {best_fold['fold']} "
      f"→ Val Loss: {best_fold['best_val_loss']:.4f} "
      f"en época {best_fold['best_epoch']}")
print(f" model_maskrcnn listo para evaluación en test")

# Guardar resultados
with open(os.path.join(MODELS_PATH, "maskrcnn_kfold_results.pkl"), "wb") as f:
    pickle.dump({"fold_results": fold_results, "best_fold": best_fold}, f)

#### **5.1.6. Validación**

In [ ]:
model_maskrcnn.eval()

test_loader_mrcnn = DataLoader(
    test_dataset_mrcnn, batch_size=1,
    shuffle=False, collate_fn=collate_fn, num_workers=0
)

all_predictions = []
all_targets     = []

print(f"{'='*60}")
print(f"PREDICCIONES EN TEST — Mask R-CNN")
print(f"Total imágenes test: {len(test_dataset_mrcnn)}")
print(f"{'='*60}")

with torch.no_grad():
    for i, (imgs, targets) in enumerate(test_loader_mrcnn):
        imgs = [img.to(DEVICE) for img in imgs]

        outputs = model_maskrcnn(imgs)

        for output, target in zip(outputs, targets):
            all_predictions.append({
                "boxes":   output["boxes"].cpu(),
                "labels":  output["labels"].cpu(),
                "scores":  output["scores"].cpu(),
                "masks":   output["masks"].cpu(),   # (N, 1, H, W) float [0,1]
            })
            all_targets.append({
                "boxes":   target["boxes"].cpu(),
                "labels":  target["labels"].cpu(),
                "masks":   target["masks"].cpu(),   # (N, H, W) binaria
            })

        print(f"  Imagen {i+1:03d}/{len(test_dataset_mrcnn)} | "
              f"Instancias predichas: {len(output['boxes'])} | "
              f"Instancias reales: {len(target['boxes'])}")

print(f"\n Predicciones completadas")
print(f"  Total imágenes procesadas : {len(all_predictions)}")
print(f"  Total imágenes con targets: {len(all_targets)}")

# Guardar predicciones y targets
with open(os.path.join(MODELS_PATH, "maskrcnn_predictions.pkl"), "wb") as f:
    pickle.dump({
        "predictions": all_predictions,
        "targets":     all_targets
    }, f)
print("Predicciones guardadas en disco ✓")

#### **1.5.7. Evaluación con métricas**

Se carga el mejor modelo guardado y se evalúa en los conjuntos de validación 
y test calculando las siete métricas requeridas: Dice, IoU, Precision, Recall, 
AUC, Hausdorff Distance y Balanced Accuracy.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

NUM_SAMPLES = 5  

sample_indices = random.sample(range(len(test_dataset_mrcnn)), NUM_SAMPLES)

fig, axes = plt.subplots(NUM_SAMPLES, 3, figsize=(15, NUM_SAMPLES * 5))
fig.suptitle("Mask R-CNN — Comparación Visual en Test",
             fontsize=16, fontweight="bold", y=1.01)

for row, idx in enumerate(sample_indices):
    img_tensor, target = test_dataset_mrcnn[idx]
    pred               = all_predictions[idx]

    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    H, W   = img_np.shape[:2]

    if len(target["masks"]) > 0:
        gt_mask = (target["masks"].sum(dim=0) > 0).numpy().astype(np.uint8)
    else:
        gt_mask = np.zeros((H, W), dtype=np.uint8)

    keep = pred["scores"] >= SCORE_THRESHOLD
    pred_masks = pred["masks"][keep]
    if len(pred_masks) > 0:
        pred_mask = (pred_masks[:, 0].sum(dim=0) > 0.5).numpy().astype(np.uint8)
    else:
        pred_mask = np.zeros((H, W), dtype=np.uint8)

    m = metrics_per_image[idx]

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"Original (img {idx})", fontsize=11)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(gt_mask, alpha=0.5, cmap="Greens")
    axes[row, 1].set_title(f"Ground Truth\n"
                            f"Instancias: {len(target['masks'])}", fontsize=11)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(img_np)
    axes[row, 2].imshow(pred_mask, alpha=0.5, cmap="Reds")
    axes[row, 2].set_title(f"Predicción\n"
                            f"Dice: {m['dice']:.3f} | IoU: {m['iou']:.3f} | "
                            f"Instancias: {keep.sum().item()}", fontsize=11)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "maskrcnn_comparacion_visual.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# Acumular gt y predicciones continuas de todas las imágenes
all_gt_flat   = []
all_pred_flat = []

for pred, tgt in zip(all_predictions, all_targets):
    H, W = tgt["masks"].shape[-2], tgt["masks"].shape[-1]

    # Ground truth binaria
    if len(tgt["masks"]) > 0:
        gt_mask = (tgt["masks"].sum(dim=0) > 0).numpy().astype(int)
    else:
        gt_mask = np.zeros((H, W), dtype=int)

    # Predicción continua
    keep = pred["scores"] >= SCORE_THRESHOLD
    pred_masks_raw = pred["masks"][keep]
    if len(pred_masks_raw) > 0:
        pred_continuous = pred_masks_raw[:, 0].sum(dim=0).numpy()
        pred_continuous = np.clip(pred_continuous, 0, 1)
    else:
        pred_continuous = np.zeros((H, W), dtype=np.float32)

    all_gt_flat.append(gt_mask.flatten())
    all_pred_flat.append(pred_continuous.flatten())

all_gt_flat   = np.concatenate(all_gt_flat)
all_pred_flat = np.concatenate(all_pred_flat)

# Curva ROC 
fpr, tpr, _ = roc_curve(all_gt_flat, all_pred_flat)
roc_auc     = auc(fpr, tpr)

# Curva Precision-Recall 
precision_curve, recall_curve, _ = precision_recall_curve(all_gt_flat, all_pred_flat)
pr_auc = auc(recall_curve, precision_curve)

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Mask R-CNN — Curvas ROC y Precision-Recall",
             fontsize=14, fontweight="bold")

# ROC
axes[0].plot(fpr, tpr, color="steelblue", lw=2,
             label=f"ROC (AUC = {roc_auc:.4f})")
axes[0].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--",
             label="Clasificador aleatorio")
axes[0].fill_between(fpr, tpr, alpha=0.1, color="steelblue")
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate", fontsize=12)
axes[0].set_title("Curva ROC", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Precision-Recall
axes[1].plot(recall_curve, precision_curve, color="darkorange", lw=2,
             label=f"PR (AUC = {pr_auc:.4f})")
axes[1].fill_between(recall_curve, precision_curve, alpha=0.1, color="darkorange")
axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("Curva Precision-Recall", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "maskrcnn_curvas_roc_pr.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print(f"   ROC  AUC : {roc_auc:.4f}")
print(f"   PR   AUC : {pr_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(NUM_SAMPLES, 4, figsize=(20, NUM_SAMPLES * 5))
fig.suptitle("Mask R-CNN — Mapas de Error en Test",
             fontsize=16, fontweight="bold", y=1.01)

for row, idx in enumerate(sample_indices):
    img_tensor, target = test_dataset_mrcnn[idx]
    pred               = all_predictions[idx]

    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    H, W   = img_np.shape[:2]

    # Ground Truth
    if len(target["masks"]) > 0:
        gt_mask = (target["masks"].sum(dim=0) > 0).numpy().astype(np.uint8)
    else:
        gt_mask = np.zeros((H, W), dtype=np.uint8)

    # Predicción
    keep = pred["scores"] >= SCORE_THRESHOLD
    pred_masks = pred["masks"][keep]
    if len(pred_masks) > 0:
        pred_mask = (pred_masks[:, 0].sum(dim=0) > 0.5).numpy().astype(np.uint8)
    else:
        pred_mask = np.zeros((H, W), dtype=np.uint8)

    # Cálculo FP y FN
    FP = ((pred_mask == 1) & (gt_mask == 0)).astype(np.uint8)  
    FN = ((pred_mask == 0) & (gt_mask == 1)).astype(np.uint8)  
    TP = ((pred_mask == 1) & (gt_mask == 1)).astype(np.uint8)  

    # Mapa de error RGB
    error_map = np.zeros((H, W, 3), dtype=np.float32)
    error_map[TP == 1] = [0.0, 0.8, 0.0]  
    error_map[FP == 1] = [0.9, 0.0, 0.0]   
    error_map[FN == 1] = [0.0, 0.0, 0.9]  

    m = metrics_per_image[idx]

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"Original (img {idx})", fontsize=11)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(gt_mask, alpha=0.5, cmap="Greens")
    axes[row, 1].set_title(f"Ground Truth\n"
                            f"Instancias: {len(target['masks'])}", fontsize=11)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(img_np)
    axes[row, 2].imshow(pred_mask, alpha=0.5, cmap="Reds")
    axes[row, 2].set_title(f"Predicción\n"
                            f"Instancias: {keep.sum().item()}", fontsize=11)
    axes[row, 2].axis("off")

    axes[row, 3].imshow(img_np)
    axes[row, 3].imshow(error_map, alpha=0.6)
    axes[row, 3].set_title(f"Mapa de Error\n"
                            f"Dice: {m['dice']:.3f} | IoU: {m['iou']:.3f}", fontsize=11)
    axes[row, 3].axis("off")

tp_patch = mpatches.Patch(color=(0.0, 0.8, 0.0), label="TP (correcto)")
fp_patch = mpatches.Patch(color=(0.9, 0.0, 0.0), label="FP (falso positivo)")
fn_patch = mpatches.Patch(color=(0.0, 0.0, 0.9), label="FN (falso negativo)")
fig.legend(handles=[tp_patch, fp_patch, fn_patch],
           loc="lower center", ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "maskrcnn_mapas_error.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
metricas_nombres = ["dice", "iou", "precision", "recall",
                    "auc", "hausdorff", "bal_acc"]
metricas_labels  = ["Dice", "IoU", "Precision", "Recall",
                    "AUC", "Hausdorff", "Balanced Acc"]

valores = {m: [img[m] for img in metrics_per_image]
           for m in metricas_nombres}

fig, axes = plt.subplots(1, 7, figsize=(22, 6))
fig.suptitle("Mask R-CNN — Boxplots de Métricas en Test",
             fontsize=14, fontweight="bold")

colors = ["steelblue", "darkorange", "green", "red",
          "purple", "brown", "teal"]

for ax, metrica, label, color in zip(axes, metricas_nombres,
                                      metricas_labels, colors):
    bp = ax.boxplot(valores[metrica], patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.7)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_xticks([])
    ax.set_ylabel("Valor", fontsize=10)
    ax.grid(alpha=0.3)
    ax.text(1, np.mean(valores[metrica]),
            f"μ={np.mean(valores[metrica]):.3f}",
            ha="center", va="bottom", fontsize=9, color="black")

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "maskrcnn_boxplots.png"),
            dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Mask R-CNN — Distribución de Métricas en Test",
             fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, (metrica, label, color) in enumerate(zip(metricas_nombres,
                                                  metricas_labels,
                                                  colors)):
    axes[i].hist(valores[metrica], bins=15, color=color,
                 alpha=0.7, edgecolor="black")
    axes[i].axvline(np.mean(valores[metrica]), color="black",
                    linestyle="--", linewidth=1.5,
                    label=f"μ={np.mean(valores[metrica]):.3f}")
    axes[i].axvline(np.median(valores[metrica]), color="red",
                    linestyle="--", linewidth=1.5,
                    label=f"med={np.median(valores[metrica]):.3f}")
    axes[i].set_title(label, fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Valor", fontsize=10)
    axes[i].set_ylabel("Frecuencia", fontsize=10)
    axes[i].legend(fontsize=9)
    axes[i].grid(alpha=0.3)

axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "maskrcnn_histogramas.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

# Métricas por imagen 
df_metrics = pd.DataFrame(metrics_per_image)
df_metrics.index = [f"Img {i+1}" for i in range(len(df_metrics))]
df_metrics.columns = ["Dice", "IoU", "Precision", "Recall",
                       "AUC", "Hausdorff", "Balanced Acc"]

# Estadísticas globales 
df_summary = pd.DataFrame({
    "Dice":         [np.mean(valores["dice"]),      np.std(valores["dice"]),
                     np.min(valores["dice"]),        np.max(valores["dice"])],
    "IoU":          [np.mean(valores["iou"]),        np.std(valores["iou"]),
                     np.min(valores["iou"]),          np.max(valores["iou"])],
    "Precision":    [np.mean(valores["precision"]),  np.std(valores["precision"]),
                     np.min(valores["precision"]),    np.max(valores["precision"])],
    "Recall":       [np.mean(valores["recall"]),     np.std(valores["recall"]),
                     np.min(valores["recall"]),       np.max(valores["recall"])],
    "AUC":          [np.mean(valores["auc"]),        np.std(valores["auc"]),
                     np.min(valores["auc"]),          np.max(valores["auc"])],
    "Hausdorff":    [np.mean(valores["hausdorff"]),  np.std(valores["hausdorff"]),
                     np.min(valores["hausdorff"]),    np.max(valores["hausdorff"])],
    "Balanced Acc": [np.mean(valores["bal_acc"]),    np.std(valores["bal_acc"]),
                     np.min(valores["bal_acc"]),      np.max(valores["bal_acc"])],
}, index=["Media", "Std", "Min", "Max"])

print(f"\n{'='*75}")
print("RESULTADOS FINALES — Mask R-CNN — Test")
print(f"{'='*75}")

print("\n Métricas por imagen:")
print(df_metrics.to_string(float_format=lambda x: f"{x:.4f}"))

print(f"\n{'─'*75}")
print("\n Resumen estadístico:")
print(df_summary.to_string(float_format=lambda x: f"{x:.4f}"))

best_img = df_metrics["Dice"].idxmax()
worst_img = df_metrics["Dice"].idxmin()
print(f"\n{'─'*75}")
print(f" Mejor imagen  : {best_img} → Dice={df_metrics.loc[best_img, 'Dice']:.4f} | "
      f"IoU={df_metrics.loc[best_img, 'IoU']:.4f}")
print(f"  Peor imagen   : {worst_img} → Dice={df_metrics.loc[worst_img, 'Dice']:.4f} | "
      f"IoU={df_metrics.loc[worst_img, 'IoU']:.4f}")

print(f"\n{'='*75}")
print("TABLA COMPARATIVA — formato proyecto")
print(f"{'='*75}")
print(f"{'Modelo':<20} {'Dice':>7} {'IoU':>7} {'Prec':>7} {'Rec':>7} "
      f"{'AUC':>7} {'Haus':>8} {'BalAcc':>8}")
print(f"{'─'*75}")
print(f"{'Mask R-CNN':<20} "
      f"{np.mean(valores['dice']):>7.4f} "
      f"{np.mean(valores['iou']):>7.4f} "
      f"{np.mean(valores['precision']):>7.4f} "
      f"{np.mean(valores['recall']):>7.4f} "
      f"{np.mean(valores['auc']):>7.4f} "
      f"{np.mean(valores['hausdorff']):>8.2f} "
      f"{np.mean(valores['bal_acc']):>8.4f}")
print(f"{'─'*75}")

df_metrics.to_csv(os.path.join(MODELS_PATH, "maskrcnn_metrics_por_imagen.csv"))
df_summary.to_csv(os.path.join(MODELS_PATH, "maskrcnn_metrics_resumen.csv"))

### **5.2. Hover-Net**

#### **5.2.1. Dataset**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2

class HoverNetDataset(Dataset):

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img_tensor, target = self.base_dataset[idx]
        H, W = img_tensor.shape[1], img_tensor.shape[2]

        # NP Map (binario: célula=1, fondo=0) 
        if len(target["masks"]) > 0:
            np_map = (target["masks"].sum(dim=0) > 0).float()
        else:
            np_map = torch.zeros(H, W)

        #  HV Map (gradientes por instancia) 
        hv_map = torch.zeros(2, H, W)  # canal 0: horizontal, canal 1: vertical

        for mask in target["masks"]:
            mask_np = mask.numpy().astype(np.uint8)
            ys, xs  = np.where(mask_np > 0)
            if len(xs) == 0:
                continue

            cx = (xs.min() + xs.max()) / 2
            cy = (ys.min() + ys.max()) / 2
            x_range = max(xs.max() - xs.min(), 1)
            y_range = max(ys.max() - ys.min(), 1)

            hv_map[0, ys, xs] = torch.tensor(
                (xs - cx) / x_range, dtype=torch.float32)
            hv_map[1, ys, xs] = torch.tensor(
                (ys - cy) / y_range, dtype=torch.float32)

        nc_map = np_map.clone()

        return img_tensor, {
            "np_map": np_map.unsqueeze(0),   
            "hv_map": hv_map,                 
            "nc_map": nc_map.unsqueeze(0),   
        }


train_dataset_hovernet = HoverNetDataset(train_dataset_mrcnn)
val_dataset_hovernet   = HoverNetDataset(val_dataset_mrcnn)
test_dataset_hovernet  = HoverNetDataset(test_dataset_mrcnn)

print(f"Dataset HoverNet creado:")
print(f"  Train : {len(train_dataset_hovernet)} imágenes")
print(f"  Val   : {len(val_dataset_hovernet)} imágenes")
print(f"  Test  : {len(test_dataset_hovernet)} imágenes")

# Verificar una muestra
img, maps = train_dataset_hovernet[0]
print(f"\n  Imagen  : {img.shape}")
print(f"  NP Map  : {maps['np_map'].shape}")
print(f"  HV Map  : {maps['hv_map'].shape}")
print(f"  NC Map  : {maps['nc_map'].shape}")

#### **5.2.2. Arquitectura**

In [ ]:
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1  = ConvBnRelu(in_ch, out_ch)
        self.conv2  = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.skip   = nn.Conv2d(in_ch, out_ch, 1, bias=False) \
                      if in_ch != out_ch else nn.Identity()
        self.relu   = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv2(self.conv1(x)) + self.skip(x))


class HoverNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ResBlock(3,   64)
        self.enc2 = ResBlock(64,  128)
        self.enc3 = ResBlock(128, 256)
        self.enc4 = ResBlock(256, 512)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        return e1, e2, e3, e4


class HoverNetDecoder(nn.Module):
    def __init__(self, out_ch):
        super().__init__()
        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3  = ResBlock(512, 256)
        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2  = ResBlock(256, 128)
        self.up1   = nn.ConvTranspose2d(128, 64,  2, stride=2)
        self.dec1  = ResBlock(128, 64)
        self.head  = nn.Conv2d(64, out_ch, 1)

    def forward(self, e1, e2, e3, e4):
        d3 = self.dec3(torch.cat([self.up3(e4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


class HoverNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder    = HoverNetEncoder()
        self.np_decoder = HoverNetDecoder(out_ch=1) 
        self.hv_decoder = HoverNetDecoder(out_ch=2)   
        self.nc_decoder = HoverNetDecoder(out_ch=1)   

    def forward(self, x):
        e1, e2, e3, e4 = self.encoder(x)
        np_pred = torch.sigmoid(self.np_decoder(e1, e2, e3, e4))
        hv_pred = self.hv_decoder(e1, e2, e3, e4)
        nc_pred = torch.sigmoid(self.nc_decoder(e1, e2, e3, e4))
        return {"np": np_pred, "hv": hv_pred, "nc": nc_pred}


def build_hovernet():
    return HoverNet()


# Verificar arquitectura
model_test = build_hovernet().to(DEVICE)
dummy      = torch.randn(1, 3, 256, 256).to(DEVICE)
out        = model_test(dummy)
print("Arquitectura HoverNet verificada:")
print(f"  NP output : {out['np'].shape}")
print(f"  HV output : {out['hv'].shape}")
print(f"  NC output : {out['nc'].shape}")
del model_test
torch.cuda.empty_cache()

#### **5.2.3. Loss functions**

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    pred   = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    inter  = (pred * target).sum()
    return 1 - (2 * inter + smooth) / (pred.sum() + target.sum() + smooth)


def mse_loss_hv(pred, target):
    return F.mse_loss(pred, target)


def hovernet_loss(outputs, targets, w_np=1.0, w_hv=2.0, w_nc=1.0):

    np_pred = outputs["np"]
    hv_pred = outputs["hv"]
    nc_pred = outputs["nc"]

    np_target = targets["np_map"].to(np_pred.device)
    hv_target = targets["hv_map"].to(hv_pred.device)
    nc_target = targets["nc_map"].to(nc_pred.device)

    # NP loss
    np_bce  = F.binary_cross_entropy(np_pred, np_target)
    np_dice = dice_loss(np_pred, np_target)
    loss_np = np_bce + np_dice

    # HV loss
    loss_hv = mse_loss_hv(hv_pred, hv_target)

    # NC loss
    nc_bce  = F.binary_cross_entropy(nc_pred, nc_target)
    nc_dice = dice_loss(nc_pred, nc_target)
    loss_nc = nc_bce + nc_dice

    total = w_np * loss_np + w_hv * loss_hv + w_nc * loss_nc
    return total, {"loss_np": loss_np.item(),
                   "loss_hv": loss_hv.item(),
                   "loss_nc": loss_nc.item()}


#### **5.2.4. Collate FN**

In [ ]:
def collate_fn_hovernet(batch):
    imgs    = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    imgs    = torch.stack(imgs, dim=0)
    return imgs, targets

#### **5.2.5.  Búsqueda de hiperparámetros**

#### **5.2.5.1. Búsqueda del mejor Learnig Rate**

In [ ]:
import pickle

train_loader_lrfinder_hv = DataLoader(
    train_dataset_hovernet, batch_size=2,
    shuffle=True, collate_fn=collate_fn_hovernet, num_workers=4,
    pin_memory=True
)

model_tmp_hv = build_hovernet().to(DEVICE)
opt_tmp_hv   = torch.optim.Adam(model_tmp_hv.parameters(),
                                 lr=1e-7, weight_decay=1e-4)

num_iter  = 100
start_lr  = 1e-7
end_lr    = 1.0
smooth_f  = 0.05
lr_mult   = (end_lr / start_lr) ** (1 / num_iter)

lr_history_hv   = []
loss_history_hv = []
best_loss_hv    = float("inf")
avg_loss_hv     = 0.0

model_tmp_hv.train()
data_iter_hv = iter(train_loader_lrfinder_hv)

print("Ejecutando LR Finder — HoverNet...")

for iteration in range(num_iter):
    try:
        imgs, targets = next(data_iter_hv)
    except StopIteration:
        data_iter_hv = iter(train_loader_lrfinder_hv)
        imgs, targets = next(data_iter_hv)

    imgs = imgs.to(DEVICE)

    opt_tmp_hv.zero_grad()
    with torch.cuda.amp.autocast():
        outputs = model_tmp_hv(imgs)
        loss, _ = hovernet_loss(outputs, {
            "np_map": torch.stack([t["np_map"] for t in targets]),
            "hv_map": torch.stack([t["hv_map"] for t in targets]),
            "nc_map": torch.stack([t["nc_map"] for t in targets]),
        })
    loss.backward()
    opt_tmp_hv.step()

    avg_loss_hv   = smooth_f * loss.item() + (1 - smooth_f) * avg_loss_hv
    smoothed_loss = avg_loss_hv / (1 - (1 - smooth_f) ** (iteration + 1))

    current_lr = opt_tmp_hv.param_groups[0]["lr"]
    lr_history_hv.append(current_lr)
    loss_history_hv.append(smoothed_loss)

    if iteration > 0 and smoothed_loss > 4 * best_loss_hv:
        print(f" Loss divergió en iteración {iteration}, deteniendo.")
        break

    if smoothed_loss < best_loss_hv:
        best_loss_hv = smoothed_loss

    for pg in opt_tmp_hv.param_groups:
        pg["lr"] *= lr_mult

    if (iteration + 1) % 10 == 0:
        print(f"  Iter {iteration+1:03d}/{num_iter} | "
              f"LR: {current_lr:.2e} | Loss: {smoothed_loss:.4f}")


skip_start = 10
skip_end   = 5
lrs_hv     = lr_history_hv[skip_start:-skip_end]
losses_hv  = loss_history_hv[skip_start:-skip_end]

min_loss_idx_hv = np.argmin(losses_hv)
optimal_idx_hv  = max(0, int(min_loss_idx_hv * 0.7))
best_lr_hv      = lrs_hv[optimal_idx_hv]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lrs_hv, losses_hv, color="darkorange", lw=2, label="Loss suavizada")
ax.axvline(best_lr_hv, color="red", linestyle="--", lw=2,
           label=f"Mejor LR: {best_lr_hv:.2e}")
ax.axvline(lrs_hv[min_loss_idx_hv], color="green", linestyle="--", lw=2,
           label=f"Mínimo Loss: {lrs_hv[min_loss_idx_hv]:.2e}")
ax.set_xscale("log")
ax.set_xlabel("Learning Rate (escala log)", fontsize=12)
ax.set_ylabel("Loss (suavizada)", fontsize=12)
ax.set_title("Learning Rate Finder — HoverNet", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "hovernet_lr_finder.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print(f"\n Mejor LR sugerido : {best_lr_hv:.2e}")
print(f"   LR en mínimo loss : {lrs_hv[min_loss_idx_hv]:.2e}")

del model_tmp_hv, train_loader_lrfinder_hv
torch.cuda.empty_cache()

with open(os.path.join(MODELS_PATH, "hovernet_lr_finder.pkl"), "wb") as f:
    pickle.dump({"best_lr":      best_lr_hv,
                 "lr_history":   lr_history_hv,
                 "loss_history": loss_history_hv}, f)
print(f" Mejor LR guardado en disco: {best_lr_hv:.2e}")

#### **5.2.5.2. Búsqueda de mejor `Batch_size`, `Optimizer`, `Scheduler`**

In [ ]:
import torch.optim as optim
from torch.utils.data import Subset
import random

hp_grid_hv = [
    {"lr": best_lr_hv, "batch_size": 2, "optimizer": "SGD",  "scheduler": "StepLR"},
    {"lr": best_lr_hv, "batch_size": 4, "optimizer": "Adam", "scheduler": "ReduceLR"},
]

# Subset 50% del train
n_hv            = len(train_dataset_hovernet)
subset_idx_hv   = random.sample(range(n_hv), int(n_hv * 0.5))
train_hp_subset_hv = Subset(train_dataset_hovernet, subset_idx_hv)

print(f"Dataset completo : {n_hv} imágenes")
print(f"Subset HP Search : {len(train_hp_subset_hv)} imágenes (50%)")

MAX_EPOCHS_HV = 10
PATIENCE_HV   = 3
hp_results_hv = []

for i, hp in enumerate(hp_grid_hv):
    print(f"\n{'='*60}")
    print(f"Experimento {i+1}/{len(hp_grid_hv)} | LR={hp['lr']:.2e} | "
          f"Batch={hp['batch_size']} | "
          f"Optimizer={hp['optimizer']} | Scheduler={hp['scheduler']}")
    print(f"{'='*60}")

    train_loader_hp_hv = DataLoader(
        train_hp_subset_hv, batch_size=hp['batch_size'],
        shuffle=True, collate_fn=collate_fn_hovernet,
        num_workers=4, pin_memory=True
    )
    val_loader_hp_hv = DataLoader(
        val_dataset_hovernet, batch_size=2,
        shuffle=False, collate_fn=collate_fn_hovernet,
        num_workers=4, pin_memory=True
    )

    model_hp_hv = build_hovernet().to(DEVICE)
    scaler_hv   = torch.cuda.amp.GradScaler()

    if hp['optimizer'] == "SGD":
        opt_hv = optim.SGD(model_hp_hv.parameters(), lr=hp['lr'],
                            momentum=0.9, weight_decay=1e-4)
    else:
        opt_hv = optim.Adam(model_hp_hv.parameters(), lr=hp['lr'],
                             weight_decay=1e-4)

    if hp['scheduler'] == "StepLR":
        sch_hv = optim.lr_scheduler.StepLR(opt_hv, step_size=2, gamma=0.5)
    else:
        sch_hv = optim.lr_scheduler.ReduceLROnPlateau(opt_hv, mode='min',
                                                       patience=1, factor=0.5)

    best_val_hv      = float("inf")
    epochs_no_improv = 0
    best_epoch_hv    = 0

    for epoch in range(MAX_EPOCHS_HV):
        model_hp_hv.train()
        train_loss = 0.0
        for imgs, targets in train_loader_hp_hv:
            imgs       = imgs.to(DEVICE)
            np_maps    = torch.stack([t["np_map"] for t in targets]).to(DEVICE)
            hv_maps    = torch.stack([t["hv_map"] for t in targets]).to(DEVICE)
            nc_maps    = torch.stack([t["nc_map"] for t in targets]).to(DEVICE)

            opt_hv.zero_grad()
            with torch.cuda.amp.autocast():
                outputs    = model_hp_hv(imgs)
                loss, _    = hovernet_loss(outputs, {
                    "np_map": np_maps,
                    "hv_map": hv_maps,
                    "nc_map": nc_maps,
                })
            scaler_hv.scale(loss).backward()
            scaler_hv.step(opt_hv)
            scaler_hv.update()
            train_loss += loss.item()
        train_loss /= len(train_loader_hp_hv)

        model_hp_hv.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, targets in val_loader_hp_hv:
                imgs    = imgs.to(DEVICE)
                np_maps = torch.stack([t["np_map"] for t in targets]).to(DEVICE)
                hv_maps = torch.stack([t["hv_map"] for t in targets]).to(DEVICE)
                nc_maps = torch.stack([t["nc_map"] for t in targets]).to(DEVICE)
                with torch.cuda.amp.autocast():
                    outputs  = model_hp_hv(imgs)
                    loss, _  = hovernet_loss(outputs, {
                        "np_map": np_maps,
                        "hv_map": hv_maps,
                        "nc_map": nc_maps,
                    })
                val_loss += loss.item()
        val_loss /= len(val_loader_hp_hv)

        if hp['scheduler'] == "StepLR":
            sch_hv.step()
        else:
            sch_hv.step(val_loss)

        if val_loss < best_val_hv:
            best_val_hv      = val_loss
            best_epoch_hv    = epoch + 1
            epochs_no_improv = 0
        else:
            epochs_no_improv += 1

        print(f"  Época {epoch+1}/{MAX_EPOCHS_HV} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Sin mejora: {epochs_no_improv}/{PATIENCE_HV}"
              + (" ✓ mejor" if epochs_no_improv == 0 else ""))

        if epochs_no_improv >= PATIENCE_HV:
            print(f"\n Early stopping en época {epoch+1}")
            break

    hp_results_hv.append({
        **hp,
        "best_val_loss": best_val_hv,
        "best_epoch":    best_epoch_hv,
        "total_epochs":  epoch + 1,
    })
    print(f" Mejor Val Loss: {best_val_hv:.4f} en época {best_epoch_hv}")

    del model_hp_hv
    torch.cuda.empty_cache()

# Resumen
print(f"\n{'='*75}")
print("RESULTADOS BÚSQUEDA DE HIPERPARÁMETROS — HoverNet")
print(f"{'='*75}")
print(f"{'Exp':>4} {'LR':>10} {'Batch':>6} {'Best Ep':>8} {'Total Ep':>9} "
      f"{'Optimizer':>10} {'Scheduler':>10} {'Val Loss':>10}")
print("-" * 75)
for i, r in enumerate(hp_results_hv):
    print(f"{i+1:>4} {r['lr']:>10.2e} {r['batch_size']:>6} "
          f"{r['best_epoch']:>8} {r['total_epochs']:>9} "
          f"{r['optimizer']:>10} {r['scheduler']:>10} "
          f"{r['best_val_loss']:>10.4f}")

best_hp_hv = min(hp_results_hv, key=lambda x: x["best_val_loss"])
print(f"\n Mejor configuración: LR={best_hp_hv['lr']:.2e} | "
      f"Batch={best_hp_hv['batch_size']} | "
      f"{best_hp_hv['optimizer']}+{best_hp_hv['scheduler']} "
      f"→ Val Loss={best_hp_hv['best_val_loss']:.4f} "
      f"en época {best_hp_hv['best_epoch']}")

with open(os.path.join(MODELS_PATH, "hovernet_hp_search.pkl"), "wb") as f:
    pickle.dump({"hp_results": hp_results_hv, "best_hp": best_hp_hv}, f)
print("HP Search HoverNet guardado en disco ✓")

#### **5.2.6. K-Fold**

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import ConcatDataset
import numpy as np

K_FOLDS_HV          = 5
MAX_EPOCHS_KFOLD_HV = 50
PATIENCE_KFOLD_HV   = 5

full_dataset_hv = ConcatDataset([train_dataset_hovernet, val_dataset_hovernet])
kf_hv           = KFold(n_splits=K_FOLDS_HV, shuffle=True, random_state=42)
all_indices_hv  = list(range(len(full_dataset_hv)))
fold_results_hv = []

print(f"\n{'='*60}")
print(f"VALIDACIÓN CRUZADA K-FOLD (k={K_FOLDS_HV}) — HoverNet")
print(f"  LR         : {best_hp_hv['lr']:.2e}")
print(f"  Batch size : {best_hp_hv['batch_size']}")
print(f"  Optimizer  : {best_hp_hv['optimizer']}")
print(f"  Scheduler  : {best_hp_hv['scheduler']}")
print(f"{'='*60}")

for fold, (train_idx, val_idx) in enumerate(kf_hv.split(all_indices_hv)):
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold+1}/{K_FOLDS_HV} | "
          f"Train: {len(train_idx)} | Val: {len(val_idx)}")
    print(f"{'─'*60}")

    train_loader_fold_hv = DataLoader(
        Subset(full_dataset_hv, train_idx),
        batch_size=best_hp_hv['batch_size'],
        shuffle=True, collate_fn=collate_fn_hovernet,
        num_workers=4, pin_memory=True
    )
    val_loader_fold_hv = DataLoader(
        Subset(full_dataset_hv, val_idx),
        batch_size=2, shuffle=False,
        collate_fn=collate_fn_hovernet,
        num_workers=4, pin_memory=True
    )

    model_fold_hv = build_hovernet().to(DEVICE)
    scaler_fold   = torch.cuda.amp.GradScaler()

    if best_hp_hv['optimizer'] == "SGD":
        opt_fold_hv = optim.SGD(model_fold_hv.parameters(),
                                 lr=best_hp_hv['lr'],
                                 momentum=0.9, weight_decay=1e-4)
    else:
        opt_fold_hv = optim.Adam(model_fold_hv.parameters(),
                                  lr=best_hp_hv['lr'], weight_decay=1e-4)

    if best_hp_hv['scheduler'] == "StepLR":
        sch_fold_hv = optim.lr_scheduler.StepLR(opt_fold_hv,
                                                  step_size=3, gamma=0.5)
    else:
        sch_fold_hv = optim.lr_scheduler.ReduceLROnPlateau(opt_fold_hv,
                                                             mode='min',
                                                             patience=2,
                                                             factor=0.5)

    best_val_fold_hv = float("inf")
    epochs_no_improv = 0
    best_epoch_fold  = 0

    for epoch in range(MAX_EPOCHS_KFOLD_HV):
        model_fold_hv.train()
        train_loss = 0.0
        for imgs, targets in train_loader_fold_hv:
            imgs    = imgs.to(DEVICE)
            np_maps = torch.stack([t["np_map"] for t in targets]).to(DEVICE)
            hv_maps = torch.stack([t["hv_map"] for t in targets]).to(DEVICE)
            nc_maps = torch.stack([t["nc_map"] for t in targets]).to(DEVICE)
            opt_fold_hv.zero_grad()
            with torch.cuda.amp.autocast():
                outputs  = model_fold_hv(imgs)
                loss, _  = hovernet_loss(outputs, {
                    "np_map": np_maps,
                    "hv_map": hv_maps,
                    "nc_map": nc_maps,
                })
            scaler_fold.scale(loss).backward()
            scaler_fold.step(opt_fold_hv)
            scaler_fold.update()
            train_loss += loss.item()
        train_loss /= len(train_loader_fold_hv)

        model_fold_hv.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, targets in val_loader_fold_hv:
                imgs    = imgs.to(DEVICE)
                np_maps = torch.stack([t["np_map"] for t in targets]).to(DEVICE)
                hv_maps = torch.stack([t["hv_map"] for t in targets]).to(DEVICE)
                nc_maps = torch.stack([t["nc_map"] for t in targets]).to(DEVICE)
                with torch.cuda.amp.autocast():
                    outputs  = model_fold_hv(imgs)
                    loss, _  = hovernet_loss(outputs, {
                        "np_map": np_maps,
                        "hv_map": hv_maps,
                        "nc_map": nc_maps,
                    })
                val_loss += loss.item()
        val_loss /= len(val_loader_fold_hv)

        if best_hp_hv['scheduler'] == "StepLR":
            sch_fold_hv.step()
        else:
            sch_fold_hv.step(val_loss)

        if val_loss < best_val_fold_hv:
            best_val_fold_hv = val_loss
            best_epoch_fold  = epoch + 1
            epochs_no_improv = 0
            torch.save(model_fold_hv.state_dict(),
                       os.path.join(MODELS_PATH,
                                    f"hovernet_fold{fold+1}.pth"))
        else:
            epochs_no_improv += 1

        print(f"  Época {epoch+1:02d}/{MAX_EPOCHS_KFOLD_HV} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Sin mejora: {epochs_no_improv}/{PATIENCE_KFOLD_HV}"
              + (" ✓" if epochs_no_improv == 0 else ""))

        if epochs_no_improv >= PATIENCE_KFOLD_HV:
            print(f"\n  ⏹ Early stopping en época {epoch+1}")
            break

    fold_results_hv.append({
        "fold":          fold + 1,
        "best_val_loss": best_val_fold_hv,
        "best_epoch":    best_epoch_fold,
        "total_epochs":  epoch + 1,
    })
    print(f"\n Fold {fold+1} | Val Loss: {best_val_fold_hv:.4f} | "
          f"Época óptima: {best_epoch_fold}/{epoch+1}")

    del model_fold_hv
    torch.cuda.empty_cache()

# Resumen K-Fold 
val_losses_hv = [r['best_val_loss'] for r in fold_results_hv]

print(f"\n{'='*60}")
print("RESUMEN K-FOLD — HoverNet")
print(f"{'='*60}")
print(f"{'Fold':>5} {'Best Ep':>8} {'Total Ep':>9} {'Val Loss':>10}")
print(f"{'─'*60}")
for r in fold_results_hv:
    print(f"  {r['fold']:>3}   {r['best_epoch']:>8}   {r['total_epochs']:>7}   "
          f"{r['best_val_loss']:>9.4f}")
print(f"\n  Media : {np.mean(val_losses_hv):.4f}")
print(f"  Std   : {np.std(val_losses_hv):.4f}")
print(f"  Min   : {np.min(val_losses_hv):.4f}")
print(f"  Max   : {np.max(val_losses_hv):.4f}")

# ── Cargar mejor fold ─────────────────────────────────────────────
best_fold_hv    = min(fold_results_hv, key=lambda x: x["best_val_loss"])
model_hovernet  = build_hovernet().to(DEVICE)
model_hovernet.load_state_dict(
    torch.load(os.path.join(MODELS_PATH,
                            f"hovernet_fold{best_fold_hv['fold']}.pth"))
)
print(f"\n🏆 Mejor fold: Fold {best_fold_hv['fold']} "
      f"→ Val Loss: {best_fold_hv['best_val_loss']:.4f} "
      f"en época {best_fold_hv['best_epoch']}")
print(f" model_hovernet listo para evaluación en test")

with open(os.path.join(MODELS_PATH, "hovernet_kfold_results.pkl"), "wb") as f:
    pickle.dump({"fold_results": fold_results_hv,
                 "best_fold":    best_fold_hv}, f)

#### **5.2.8. Predicciones en test**

In [ ]:
model_hovernet.eval()

test_loader_hv = DataLoader(
    test_dataset_hovernet, batch_size=1,
    shuffle=False, collate_fn=collate_fn_hovernet,
    num_workers=4, pin_memory=True
)

all_predictions_hv = []
all_targets_hv     = []

print(f"{'='*60}")
print(f"PREDICCIONES EN TEST — HoverNet")
print(f"Total imágenes test: {len(test_dataset_hovernet)}")
print(f"{'='*60}")

with torch.no_grad():
    for i, (imgs, targets) in enumerate(test_loader_hv):
        imgs = imgs.to(DEVICE)
        with torch.cuda.amp.autocast():
            outputs = model_hovernet(imgs)

        all_predictions_hv.append({
            "np": outputs["np"].cpu(),
            "hv": outputs["hv"].cpu(),
            "nc": outputs["nc"].cpu(),
        })
        all_targets_hv.append(targets[0])

        print(f"  Imagen {i+1:03d}/{len(test_dataset_hovernet)} | "
              f"NP shape: {outputs['np'].shape} | "
              f"HV shape: {outputs['hv'].shape}")

print(f"\n Predicciones completadas: {len(all_predictions_hv)} imágenes")

with open(os.path.join(MODELS_PATH, "hovernet_predictions.pkl"), "wb") as f:
    pickle.dump({"predictions": all_predictions_hv,
                 "targets":     all_targets_hv}, f)

#### **5.2.9. Métricas en test**

In [ ]:
from scipy.spatial.distance import directed_hausdorff
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

THRESHOLD_HV = 0.5
metrics_per_image_hv = []

print(f"{'='*60}")
print("MÉTRICAS EN TEST — HoverNet")
print(f"{'='*60}")
print(f"{'Img':>4} {'Dice':>7} {'IoU':>7} {'Prec':>7} "
      f"{'Rec':>7} {'AUC':>7} {'Haus':>8} {'BalAcc':>8}")
print(f"{'─'*60}")

for i, (pred, tgt) in enumerate(zip(all_predictions_hv, all_targets_hv)):
    # Predicción binaria del NP map
    np_pred_cont = pred["np"][0, 0].numpy()
    np_pred_bin  = (np_pred_cont >= THRESHOLD_HV).astype(np.uint8)

    # Ground truth binaria
    np_gt = (tgt["np_map"][0] > 0).numpy().astype(np.uint8)

    # Métricas
    dice = dice_coefficient(np_gt, np_pred_bin)
    iou  = iou_score(np_gt, np_pred_bin)
    prec = precision_score_mask(np_gt, np_pred_bin)
    rec  = recall_score_mask(np_gt, np_pred_bin)
    haus = hausdorff_distance(np_gt, np_pred_bin)
    bal  = balanced_acc(np_gt, np_pred_bin)

    gt_flat   = np_gt.flatten().astype(int)
    pred_flat = np_pred_cont.flatten()
    try:
        auc = roc_auc_score(gt_flat, pred_flat) \
              if len(np.unique(gt_flat)) > 1 else 1.0
    except:
        auc = 1.0

    metrics_per_image_hv.append({
        "dice":      dice,
        "iou":       iou,
        "precision": prec,
        "recall":    rec,
        "auc":       auc,
        "hausdorff": haus,
        "bal_acc":   bal,
    })

    print(f"{i+1:>4} {dice:>7.4f} {iou:>7.4f} {prec:>7.4f} "
          f"{rec:>7.4f} {auc:>7.4f} {haus:>8.2f} {bal:>8.4f}")

# ── Resumen ───────────────────────────────────────────────────────
print(f"\n{'─'*60}")
for stat_name, fn in [("Media", np.mean), ("Std",  np.std),
                       ("Min",   np.min),  ("Max",  np.max)]:
    print(f"{stat_name:>5} "
          f"{fn([m['dice']      for m in metrics_per_image_hv]):>7.4f} "
          f"{fn([m['iou']       for m in metrics_per_image_hv]):>7.4f} "
          f"{fn([m['precision'] for m in metrics_per_image_hv]):>7.4f} "
          f"{fn([m['recall']    for m in metrics_per_image_hv]):>7.4f} "
          f"{fn([m['auc']       for m in metrics_per_image_hv]):>7.4f} "
          f"{fn([m['hausdorff'] for m in metrics_per_image_hv]):>8.2f} "
          f"{fn([m['bal_acc']   for m in metrics_per_image_hv]):>8.4f}")

with open(os.path.join(MODELS_PATH, "hovernet_test_metrics.pkl"), "wb") as f:
    pickle.dump(metrics_per_image_hv, f)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

NUM_SAMPLES_HV = 5
sample_indices_hv = random.sample(range(len(test_dataset_hovernet)), NUM_SAMPLES_HV)
valores_hv = {m: [img[m] for img in metrics_per_image_hv]
              for m in ["dice","iou","precision","recall","auc","hausdorff","bal_acc"]}

# ── Celda 10: Comparación visual ─────────────────────────────────
fig, axes = plt.subplots(NUM_SAMPLES_HV, 3, figsize=(15, NUM_SAMPLES_HV * 5))
fig.suptitle("HoverNet — Comparación Visual en Test",
             fontsize=16, fontweight="bold", y=1.01)

for row, idx in enumerate(sample_indices_hv):
    img_tensor, tgt = test_dataset_hovernet[idx]
    pred            = all_predictions_hv[idx]

    img_np   = img_tensor.permute(1, 2, 0).numpy()
    img_np   = np.clip(img_np, 0, 1)
    gt_mask  = (tgt["np_map"][0] > 0).numpy().astype(np.uint8)
    pred_map = (pred["np"][0, 0].numpy() >= THRESHOLD_HV).astype(np.uint8)
    m        = metrics_per_image_hv[idx]

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"Original (img {idx})", fontsize=11)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(gt_mask, alpha=0.5, cmap="Greens")
    axes[row, 1].set_title("Ground Truth", fontsize=11)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(img_np)
    axes[row, 2].imshow(pred_map, alpha=0.5, cmap="Reds")
    axes[row, 2].set_title(f"Predicción\nDice: {m['dice']:.3f} | "
                            f"IoU: {m['iou']:.3f}", fontsize=11)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "hovernet_comparacion_visual.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(" Comparación visual guardada ✓")

# ── Celda 11: Curvas ROC y PR ─────────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve, auc

all_gt_hv   = []
all_pred_hv = []

for pred, tgt in zip(all_predictions_hv, all_targets_hv):
    all_gt_hv.append((tgt["np_map"][0] > 0).numpy().astype(int).flatten())
    all_pred_hv.append(pred["np"][0, 0].numpy().flatten())

all_gt_hv   = np.concatenate(all_gt_hv)
all_pred_hv = np.concatenate(all_pred_hv)

fpr_hv, tpr_hv, _ = roc_curve(all_gt_hv, all_pred_hv)
roc_auc_hv         = auc(fpr_hv, tpr_hv)
prec_hv, rec_hv, _ = precision_recall_curve(all_gt_hv, all_pred_hv)
pr_auc_hv           = auc(rec_hv, prec_hv)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("HoverNet — Curvas ROC y Precision-Recall",
             fontsize=14, fontweight="bold")

axes[0].plot(fpr_hv, tpr_hv, color="darkorange", lw=2,
             label=f"ROC (AUC={roc_auc_hv:.4f})")
axes[0].plot([0,1],[0,1], color="gray", lw=1, linestyle="--")
axes[0].fill_between(fpr_hv, tpr_hv, alpha=0.1, color="darkorange")
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate", fontsize=12)
axes[0].set_title("Curva ROC", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].plot(rec_hv, prec_hv, color="steelblue", lw=2,
             label=f"PR (AUC={pr_auc_hv:.4f})")
axes[1].fill_between(rec_hv, prec_hv, alpha=0.1, color="steelblue")
axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("Curva Precision-Recall", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "hovernet_curvas_roc_pr.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ ROC AUC: {roc_auc_hv:.4f} | PR AUC: {pr_auc_hv:.4f}")

# ── Celda 12: Mapas de error ──────────────────────────────────────
fig, axes = plt.subplots(NUM_SAMPLES_HV, 4, figsize=(20, NUM_SAMPLES_HV * 5))
fig.suptitle("HoverNet — Mapas de Error",
             fontsize=16, fontweight="bold", y=1.01)

for row, idx in enumerate(sample_indices_hv):
    img_tensor, tgt = test_dataset_hovernet[idx]
    pred            = all_predictions_hv[idx]

    img_np   = img_tensor.permute(1, 2, 0).numpy()
    img_np   = np.clip(img_np, 0, 1)
    gt_mask  = (tgt["np_map"][0] > 0).numpy().astype(np.uint8)
    pred_map = (pred["np"][0, 0].numpy() >= THRESHOLD_HV).astype(np.uint8)

    H, W      = img_np.shape[:2]
    FP        = ((pred_map == 1) & (gt_mask == 0)).astype(np.uint8)
    FN        = ((pred_map == 0) & (gt_mask == 1)).astype(np.uint8)
    TP        = ((pred_map == 1) & (gt_mask == 1)).astype(np.uint8)
    error_map = np.zeros((H, W, 3), dtype=np.float32)
    error_map[TP == 1] = [0.0, 0.8, 0.0]
    error_map[FP == 1] = [0.9, 0.0, 0.0]
    error_map[FN == 1] = [0.0, 0.0, 0.9]

    m = metrics_per_image_hv[idx]

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"Original (img {idx})", fontsize=11)
    axes[row, 0].axis("off")
    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(gt_mask, alpha=0.5, cmap="Greens")
    axes[row, 1].set_title("Ground Truth", fontsize=11)
    axes[row, 1].axis("off")
    axes[row, 2].imshow(img_np)
    axes[row, 2].imshow(pred_map, alpha=0.5, cmap="Reds")
    axes[row, 2].set_title("Predicción", fontsize=11)
    axes[row, 2].axis("off")
    axes[row, 3].imshow(img_np)
    axes[row, 3].imshow(error_map, alpha=0.6)
    axes[row, 3].set_title(f"Mapa de Error\n"
                            f"Dice: {m['dice']:.3f} | IoU: {m['iou']:.3f}",
                            fontsize=11)
    axes[row, 3].axis("off")

tp_patch = mpatches.Patch(color=(0.0, 0.8, 0.0), label="TP")
fp_patch = mpatches.Patch(color=(0.9, 0.0, 0.0), label="FP")
fn_patch = mpatches.Patch(color=(0.0, 0.0, 0.9), label="FN")
fig.legend(handles=[tp_patch, fp_patch, fn_patch],
           loc="lower center", ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "hovernet_mapas_error.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(" Mapas de error guardados ✓")

# ── Celda 13: Boxplots e Histogramas ─────────────────────────────
metricas_nombres = ["dice","iou","precision","recall","auc","hausdorff","bal_acc"]
metricas_labels  = ["Dice","IoU","Precision","Recall","AUC","Hausdorff","Balanced Acc"]
colors           = ["steelblue","darkorange","green","red","purple","brown","teal"]

fig, axes = plt.subplots(1, 7, figsize=(22, 6))
fig.suptitle("HoverNet — Boxplots de Métricas en Test",
             fontsize=14, fontweight="bold")

for ax, metrica, label, color in zip(axes, metricas_nombres,
                                      metricas_labels, colors):
    bp = ax.boxplot(valores_hv[metrica], patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.7)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_xticks([])
    ax.grid(alpha=0.3)
    ax.text(1, np.mean(valores_hv[metrica]),
            f"μ={np.mean(valores_hv[metrica]):.3f}",
            ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "hovernet_boxplots.png"),
            dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("HoverNet — Distribución de Métricas en Test",
             fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, (metrica, label, color) in enumerate(zip(metricas_nombres,
                                                  metricas_labels, colors)):
    axes[i].hist(valores_hv[metrica], bins=15, color=color,
                 alpha=0.7, edgecolor="black")
    axes[i].axvline(np.mean(valores_hv[metrica]), color="black",
                    linestyle="--", lw=1.5,
                    label=f"μ={np.mean(valores_hv[metrica]):.3f}")
    axes[i].axvline(np.median(valores_hv[metrica]), color="red",
                    linestyle="--", lw=1.5,
                    label=f"med={np.median(valores_hv[metrica]):.3f}")
    axes[i].set_title(label, fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Valor", fontsize=10)
    axes[i].set_ylabel("Frecuencia", fontsize=10)
    axes[i].legend(fontsize=9)
    axes[i].grid(alpha=0.3)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_PATH, "hovernet_histogramas.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ── Tabla resumen ─────────────────────────────────────────────────
import pandas as pd

df_summary_hv = pd.DataFrame({
    "Dice":         [np.mean(valores_hv["dice"]),      np.std(valores_hv["dice"])],
    "IoU":          [np.mean(valores_hv["iou"]),        np.std(valores_hv["iou"])],
    "Precision":    [np.mean(valores_hv["precision"]),  np.std(valores_hv["precision"])],
    "Recall":       [np.mean(valores_hv["recall"]),     np.std(valores_hv["recall"])],
    "AUC":          [np.mean(valores_hv["auc"]),        np.std(valores_hv["auc"])],
    "Hausdorff":    [np.mean(valores_hv["hausdorff"]),  np.std(valores_hv["hausdorff"])],
    "Balanced Acc": [np.mean(valores_hv["bal_acc"]),    np.std(valores_hv["bal_acc"])],
}, index=["Media", "Std"])

print(f"\n{'='*75}")
print("TABLA RESUMEN — HoverNet — Test")
print(f"{'='*75}")
print(df_summary_hv.to_string(float_format=lambda x: f"{x:.4f}"))

df_summary_hv.to_csv(os.path.join(MODELS_PATH, "hovernet_metrics_resumen.csv"))